# Probar el modo empresa en Kaggle

Este notebook corre la empresa (26 agentes + orquestador LangGraph) en la máquina virtual gratuita de Kaggle. No necesitás computadora propia: solo el celular para subir este archivo a Kaggle.

**Antes de correr, una sola vez:**

1. Subí este archivo a Kaggle: *Create → New Notebook → Upload* (o pegá las celdas en un notebook nuevo).
2. En el notebook: *Add-ons → Secrets* → agregá dos secretos:
   - `github`: token de GitHub (o el nombre que le hayas puesto, se edita en la celda 3) con acceso al repo privado `company-os` (en github.com: *Settings → Developer settings → Personal access tokens → Generate new token (classic)*, tildá `repo`).
   - `LLM_API_KEY`: tu API key del proveedor de LLM (o el nombre que le hayas puesto, se edita en la celda 3).
3. Verificá que el notebook tenga internet: *Settings → Internet → ON*.
4. La **celda 5** ya viene configurada para NVIDIA con los modelos nuevos; la **celda 6** lista en vivo qué modelos están disponibles hoy con tu key (el catálogo rota).

Después corrés las celdas en orden. La prueba rápida (celda 7) gasta centavos; el modo empresa (celda 9) hace ~21 llamadas al modelo.

In [ ]:
# Celda 2: dependencias
!pip install -q openai pyyaml python-dotenv langgraph

In [ ]:
# Celda 3: secretos (Add-ons → Secrets en Kaggle)
# Si tus secretos tienen otros nombres, editalos acá. Para la key de NVIDIA
# también se prueban automáticamente: nvidia, NVIDIA_API_KEY, NVAPI_KEY, nvapi.
SECRETO_GITHUB = "github"
SECRETO_LLM = "NVIDIA_API_KEY"

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def _obtener(candidatos):
    for nombre in candidatos:
        try:
            v = secrets.get_secret(nombre)
            if v:
                return nombre, v
        except Exception:
            continue
    return None, None

GITHUB_TOKEN = secrets.get_secret(SECRETO_GITHUB)
assert GITHUB_TOKEN, f"Falta el secreto '{SECRETO_GITHUB}' (Add-ons → Secrets)"
_nom_llm, LLM_API_KEY = _obtener([SECRETO_LLM, "nvidia", "NVIDIA_API_KEY",
                                   "NVAPI_KEY", "nvapi", "API_KEY", "llm"])
assert LLM_API_KEY, "No encontré tu key de NVIDIA en los secretos de Kaggle"
print(f"Secretos OK (GitHub: '{SECRETO_GITHUB}', NVIDIA: '{_nom_llm}').")

In [ ]:
# Celda 4: clonar el repo privado (el token no queda guardado en el repo clonado)
import subprocess, base64, os

if not os.path.exists("company-os"):
    auth = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
    subprocess.run(
        ["git", "-c", f"http.extraHeader=AUTHORIZATION: basic {auth}",
         "clone", "--depth", "1", "https://github.com/enrrutador/company-os.git"],
        check=True, capture_output=True)
    print("Repo clonado.")
else:
    print("El repo ya estaba clonado.")

In [ ]:
# Celda 5: elegir modelos EN VIVO en NVIDIA (el catálogo rota; los IDs fijos mueren)
import os
from openai import OpenAI

_probe = OpenAI(base_url="https://integrate.api.nvidia.com/v1",
                api_key=os.environ["LITELLM_MASTER_KEY"], timeout=60)
try:
    _disp = {m.id for m in _probe.models.list()}
    print(f"({len(_disp)} modelos publicados)")
except Exception:
    _disp = set()

def _sirve(mid):
    try:
        r = _probe.chat.completions.create(
            model=mid,
            messages=[{"role": "user", "content": "Responde con una sola palabra: ok"}],
            max_tokens=256, temperature=0)
        return bool((r.choices[0].message.content or "").strip())
    except Exception:
        return False

_CANDS = {
    "MODELO_EMPRESA_BASE": [
        "deepseek-ai/deepseek-v4-flash-0731", "moonshotai/kimi-k2.6",
        "z-ai/glm5", "minimaxai/minimax-m2.5",
        "nvidia/nemotron-3-super-120b-a12b"],
    "MODELO_EMPRESA_RAZONAMIENTO": [
        "deepseek-ai/deepseek-v4-pro-0813", "deepseek-ai/deepseek-v4-pro",
        "moonshotai/kimi-k3", "nvidia/nemotron-3-ultra-550b-a55b",
        "openai/gpt-oss-120b", "qwen/qwen3.5-397b-a17b"],
    "MODELO_EMPRESA_LIGERO": [
        "nvidia/nemotron-3-nano-30b-a3b", "nvidia/nemotron-3.5-lightning-30b-a3b",
        "openai/gpt-oss-20b"],
}
os.environ["LITELLM_BASE_URL"] = "https://integrate.api.nvidia.com/v1"
os.environ["LITELLM_MASTER_KEY"] = LLM_API_KEY
for _var, _ids in _CANDS.items():
    for _mid in _ids:
        if _disp and _mid not in _disp:
            continue
        print(f"probando {_mid}...", end=" ", flush=True)
        if _sirve(_mid):
            os.environ[_var] = _mid
            print("OK")
            break
        print("no")
    assert os.environ.get(_var), f"ningún candidato funcionó para {_var}"
print("Modelos elegidos:", os.environ["MODELO_EMPRESA_BASE"], "|",
      os.environ["MODELO_EMPRESA_RAZONAMIENTO"], "|", os.environ["MODELO_EMPRESA_LIGERO"])

In [ ]:
# Celda 6: qué modelos están disponibles HOY con tu key (el catálogo rota seguido)
from openai import OpenAI

probe = OpenAI(base_url=os.environ["LITELLM_BASE_URL"],
               api_key=os.environ["LITELLM_MASTER_KEY"])
ids = sorted(m.id for m in probe.models.list())
print(f"{len(ids)} modelos disponibles:")
for i in ids:
    print(" -", i)

## Prueba rápida: un solo agente

Si esto responde, todo el cableado funciona (repo → prompt → tu proveedor → auditoría).

In [ ]:
# Celda 7: smoke test
!cd company-os/runtime && python3 ejecutar.py conciliador "Confirmá en una línea que estás corriendo en Kaggle." 2>&1 | tail -6

## Modo empresa: pipeline punta a punta

El Gerente General orquesta tesis → validación → construcción → comercial con tu objetivo en lenguaje natural. Tarda unos minutos (~28 llamadas). El despliegue frena solo: queda `PENDIENTE_APROBACION_FABIAN` hasta que vos lo apruebes.
**Nota:** si un modelo de razonamiento devuelve la respuesta vacía, es un tema conocido de algunos modelos NIM con clientes OpenAI simples: cambialo en la celda 5 por otro de la lista de la celda 6.


In [ ]:
# Celda: ═══ ESCRIBÍ TU OBJETIVO ═══
import os
os.environ["OBJETIVO"] = "Escribí acá tu objetivo en lenguaje natural"
print("Objetivo:", os.environ["OBJETIVO"])

In [ ]:
# Celda 10: correr el modo empresa
!cd company-os/runtime && python3 ejecutar_pipeline.py producto --objetivo "$OBJETIVO" 2>&1 | tail -45

## Ver la auditoría

Cada ejecución de cada agente queda registrada con modelo y tokens.

In [ ]:
# Celda 12: auditoría
!cd company-os/runtime && python3 -c "
import json
lineas = open('registro/auditoria.jsonl', encoding='utf-8').read().strip().split(chr(10))
print(f'{len(lineas)} ejecuciones auditadas')
for l in lineas[-10:]:
    d = json.loads(l)
    print('-', d['agente'], '|', d['modelo'])
" 2>&1 | tail -14